Casus: Ter voorbereiding van een machine learning project voor de voorspelling van kijkcijfers voor Vlaamse televisieprogramma's zijn drie databronnen nodig: 
- de gegevens van de kijkcijfers zelf, gescrapet van de website van de cim. Zie bijv. https://www.cim.be/nl/televisie?type=daily&date=2026-4-9&region=north. In de praktijk zal gebruik gemaakt worden van de API van de website: 
- de omzetting van de combinatie (zender, programmanaam) naar een bepaald programmagenre. Hiervoor zal gebruik gemaakt worden van de API van de LLM Groq (API key aan te vragen via https://console.groq.com/keys)
- de weergegevens (neerslag en temperatuur)

Onderstaand stukje code toont hoe de kijkcijfers voor één dag (vandaag -30 dagen) kunnen opgehaald worden. 

### Opgave 1
Integreer dit stuk code (en pas indien nodig aan) in de functie KijkCijfers::scrape_data om de kijkcijfers voor de volledige in de functie main() opgegeven periode op te halen. 

In [3]:
import datetime
import pytz
import time
import os
import sys
import requests
import json
import pandas as pd
pd.set_option("display.width", 140)

current_date = datetime.date.today() - datetime.timedelta(days=30)
url = "https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={date}&reportType=north"
year = current_date.year
date_str = current_date.strftime("%Y-%m-%d")
current_url = url.format(date=date_str)
records = []
print(f"Fetching data for {date_str}...")
try:
    response = requests.get(current_url)
    if response.status_code == 200:
        data = response.json()
        data = data.get('hydra:member')
        for item in data:
            record = {
                'ranking': item.get('ranking'),
                'description': item.get('description').upper(),
                'channel': item.get('channel').upper(),
                'dateDiff': item.get('dateDiff'),
                'startTime': item.get('startTime'),
                'rLength': item.get('rLength'),
                'rateInk': item.get('rateInK')
            }
            for key in record:
                val = record[key]
                if type(val) == str:
                    record[key] = val.replace('"', '')  # remove double quotes

            if len(record['description']) > 0: # only keep records with a description
                records.append(record)
    else:
        print(f"Request failed with status code {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")

df = pd.DataFrame(records)
print(df)

Fetching data for 2026-05-13...
   ranking         description     channel                    dateDiff startTime   rLength  rateInk
0        1               THUIS       VRT 1  2026-05-13T00:00:00.000000  20:20:05  00:23:24  887.806
1        2  HET 7 UUR-JOURNAAL       VRT 1  2026-05-13T00:00:00.000000  19:00:03  00:46:38  795.087
2        3    HOMO UNIVERSALIS       VRT 1  2026-05-13T00:00:00.000000  19:48:43  00:22:14  741.160
3        4                PANO       VRT 1  2026-05-13T00:00:00.000000  20:43:47  00:37:26  560.912
4        5        HUIS GEMAAKT         VTM  2026-05-13T00:00:00.000000  20:39:53  00:55:39  547.752
5        6      NIEUWS 19U VTM         VTM  2026-05-13T00:00:00.000000  18:59:51  00:57:04  522.989
6        7             BLOKKEN       VRT 1  2026-05-13T00:00:00.000000  18:29:30  00:27:54  518.555
7        8             FAMILIE         VTM  2026-05-13T00:00:00.000000  20:07:15  00:25:42  514.876
8        9  HET 1 UUR-JOURNAAL       VRT 1  2026-05-13T00:00:00.0000

### Opgave 2

- Gaan naar https://openmeteo.com en haal de Python-code op om de historische weerdata (temperatuur 2m en neerslag) op te halen voor een bepaalde periode op je actuele locatie. 

- Integreer deze code in de functie Weather::get_weather()

- Zorg ervoor dat je tot 5 maal probeert als de API niet beschikbaar is of andere fouten geeft. Wacht telkens 5 minuten tussen 2 pogingen. Hoe ga je dit testen? 

### Opgave 3

Voeg foutafhandeling toe. Denk na over waar fouten kunnen optreden en handel ze correct af. Maak gebruik van de klasse in logger.py. 

